<a href="https://colab.research.google.com/github/FlyRank-Internship/week1-flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlyRank-Internship/week1-flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

### Method choice

I will use clustering as the main modeling method because this lane is focused on grouping content into similar performance and refresh profiles rather than predicting a single outcome.

The clustering will use available content and performance features to identify groups with different characteristics, such as impressions, CTR, position, engagement, trend, and freshness.

I will use a decision tree as a secondary comparison and interpretation tool to examine which features help distinguish the resulting groups. The tree will be treated as decision-support rather than as evidence that one method is automatically better.

The goal is to find useful and interpretable content groups that can support different review or refresh actions, while avoiding unnecessary model complexity.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/FlyRank-Internship/week1-flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [ ]:
print(df.shape)
df.info()

(30000, 44)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30

## 2. Split design

*I will use a client-grouped split so that content from the same client does not appear in both groups. This reduces the risk that client-specific patterns make the clustering look stronger than it really is. I will use the split as a robustness check rather than treating clustering performance like supervised prediction.*

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Use client_id as the grouping variable
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

df_train = df.iloc[train_idx].copy()
df_test = df.iloc[test_idx].copy()

print("Total rows:", len(df))
print("Train rows:", len(df_train))
print("Test rows:", len(df_test))

print("Train clients:", df_train["client_id"].nunique())
print("Test clients:", df_test["client_id"].nunique())

# Confirm there is no client overlap
overlap = set(df_train["client_id"]) & set(df_test["client_id"])
print("Client overlap:", len(overlap))

Total rows: 30000
Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0


###3. Train + compare vs my baseline

I will first use K-Means clustering to identify content archetypes using non-leaky content and performance features. I will then use a shallow Decision Tree as the supervised comparison model.

The Decision Tree will use the same client-grouped split and the same declining-content target used by the reference workflow. I will compare the Decision Tree with the Week-4 baseline using the same evaluation metric, while treating the clustering results as an additional descriptive analysis rather than forcing an inappropriate predictive metric onto the clusters.

In [ ]:
# Section 3A — Prepare non-leaky features for clustering

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Features chosen to describe content performance/freshness.
# IDs and label-derived trend fields are intentionally excluded.
cluster_features = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "days_since_last_update",
    "word_count",
]

# Work with copies so the original df remains unchanged
X_train_raw = df_train[cluster_features].copy()
X_test_raw = df_test[cluster_features].copy()

# avg_position = 0 means "no data", so treat it as missing for modeling
for data in [X_train_raw, X_test_raw]:
    data["avg_position"] = data["avg_position"].replace(0, float("nan"))

# Add missingness indicators.
# This avoids hiding useful missing-data patterns with zero imputation.
for col in cluster_features:
    if col == "avg_position":
        continue
    X_train_raw[f"{col}_missing"] = X_train_raw[col].isna().astype(int)
    X_test_raw[f"{col}_missing"] = X_test_raw[col].isna().astype(int)

# The original feature list plus missingness indicators
model_features = list(X_train_raw.columns)

# Median imputation learned ONLY from training clients
imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train_raw)
X_test_imputed = imputer.transform(X_test_raw)

# Standardize using training data only
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Training rows:", X_train_scaled.shape[0])
print("Test rows:", X_test_scaled.shape[0])
print("Number of clustering features:", X_train_scaled.shape[1])
print("Missing values after imputation:",
      int(pd.DataFrame(X_train_scaled).isna().sum().sum()))

Training rows: 23837
Test rows: 6163
Number of clustering features: 13
Missing values after imputation: 0


In [ ]:
# Section 3B — Choose number of clusters using silhouette score

silhouette_results = []

for k in range(2, 7):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_train_scaled)

    score = silhouette_score(X_train_scaled, labels)

    silhouette_results.append({
        "k": k,
        "silhouette_score": score
    })

silhouette_df = pd.DataFrame(silhouette_results)

print(silhouette_df)

best_k = int(
    silhouette_df.loc[
        silhouette_df["silhouette_score"].idxmax(),
        "k"
    ]
)

print("\nSelected k:", best_k)

   k  silhouette_score
0  2          0.224368
1  3          0.240430
2  4          0.298269
3  5          0.311419
4  6          0.320717

Selected k: 6


In [ ]:
# Section 3C — Fit final K-Means model

kmeans_final = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

# Fit only on the training clients
train_clusters = kmeans_final.fit_predict(X_train_scaled)

# Assign the learned cluster model to the held-out clients
test_clusters = kmeans_final.predict(X_test_scaled)

# Add cluster labels to copies of our data
df_train_model = df_train.copy()
df_test_model = df_test.copy()

df_train_model["cluster"] = train_clusters
df_test_model["cluster"] = test_clusters

print("Number of clusters:", best_k)

print("\nTraining cluster sizes:")
print(df_train_model["cluster"].value_counts().sort_index())

print("\nTest cluster sizes:")
print(df_test_model["cluster"].value_counts().sort_index())

Number of clusters: 6

Training cluster sizes:
cluster
0     3611
1    13250
2       15
3      350
4      129
5     6482
Name: count, dtype: int64

Test cluster sizes:
cluster
0     208
1    4666
2     110
3     119
4       3
5    1057
Name: count, dtype: int64


In [ ]:
# Section 3D — Profile the clusters

profile_features = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "days_since_last_update",
    "word_count"
]

cluster_profile = (
    df_train_model
    .groupby("cluster")[profile_features]
    .mean()
    .round(2)
)

cluster_profile["n"] = (
    df_train_model["cluster"]
    .value_counts()
    .sort_index()
)

display(cluster_profile)

,impressions_90d,ctr,avg_position,engagement_rate,scroll_rate,days_since_last_update,word_count,n
cluster,,,,,,,,
0,14225.17,0.31,19.82,1.54,6.70,104.92,5528.97,3611
1,3455.93,0.39,14.52,1.64,23.59,29.41,2649.43,13250
2,96.07,0.00,15.83,6.67,NaN,31.33,3272.62,15
3,629.54,1.68,18.15,54.97,58.32,38.61,2220.99,350
4,4.22,41.60,6.16,6.28,20.68,37.07,1200.87,129
5,5090.85,0.20,18.84,1.66,7.85,58.82,1279.29,6482


In [ ]:
# Check missingness in the original features by cluster

missing_by_cluster = (
    df_train_model
    .groupby("cluster")[profile_features]
    .apply(lambda x: x.isna().mean() * 100)
    .round(2)
)

display(missing_by_cluster)

,impressions_90d,ctr,avg_position,engagement_rate,scroll_rate,days_since_last_update,word_count
cluster,,,,,,,
0,0.0,0.0,0.0,0.0,0.0,0.0,1.27
1,0.0,0.0,0.0,0.0,0.0,0.0,0.01
2,0.0,0.0,0.0,0.0,100.0,0.0,13.33
3,0.0,0.0,0.0,0.0,0.0,0.0,23.71
4,0.0,0.0,0.0,0.0,0.0,0.0,5.43
5,0.0,0.0,0.0,0.0,0.0,0.0,99.89


In [ ]:
print("Target-related columns:")
print([c for c in df.columns if "label" in c.lower() or "trend" in c.lower()])

print("\nBaseline-related columns:")
print([c for c in df.columns if "baseline" in c.lower() or "score" in c.lower()])

Target-related columns:
['trend_direction', 'trend_pct']

Baseline-related columns:
[]


In [ ]:
# Section 3E — Create the supervised target

# Target: whether the content is declining
df_train_model["is_declining_label"] = (
    df_train_model["trend_direction"] == "down"
).astype(int)

df_test_model["is_declining_label"] = (
    df_test_model["trend_direction"] == "down"
).astype(int)

print("Training label distribution:")
print(df_train_model["is_declining_label"].value_counts())

print("\nTraining declining rate:",
      round(df_train_model["is_declining_label"].mean(), 4))

print("\nTest label distribution:")
print(df_test_model["is_declining_label"].value_counts())

print("\nTest declining rate:",
      round(df_test_model["is_declining_label"].mean(), 4))

Training label distribution:
is_declining_label
1    13113
0    10724
Name: count, dtype: int64

Training declining rate: 0.5501

Test label distribution:
is_declining_label
1    3149
0    3014
Name: count, dtype: int64

Test declining rate: 0.511


In [ ]:
# Section 3F — Train a shallow Decision Tree

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

tree_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "days_since_last_update",
    "content_age_days",
    "word_count"
]

X_train_tree = df_train_model[tree_features].copy()
X_test_tree = df_test_model[tree_features].copy()

y_train = df_train_model["is_declining_label"]
y_test = df_test_model["is_declining_label"]

# avg_position = 0 means no search-position data
X_train_tree["avg_position"] = X_train_tree["avg_position"].replace(0, float("nan"))
X_test_tree["avg_position"] = X_test_tree["avg_position"].replace(0, float("nan"))

# Median imputation learned only from training data
tree_imputer = SimpleImputer(strategy="median")

X_train_tree = tree_imputer.fit_transform(X_train_tree)
X_test_tree = tree_imputer.transform(X_test_tree)

# Shallow tree: transparent and easy to inspect
tree = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

tree.fit(X_train_tree, y_train)

tree_pred = tree.predict(X_test_tree)
tree_prob = tree.predict_proba(X_test_tree)[:, 1]

print("Decision Tree trained successfully.")
print("Test accuracy:", round(accuracy_score(y_test, tree_pred), 4))
print("Test precision:", round(precision_score(y_test, tree_pred), 4))
print("Test recall:", round(recall_score(y_test, tree_pred), 4))
print("Test F1:", round(f1_score(y_test, tree_pred), 4))

Decision Tree trained successfully.
Test accuracy: 0.5776
Test precision: 0.5855
Test recall: 0.5938
Test F1: 0.5896


In [ ]:
# Section 3G — Inspect the Decision Tree

from sklearn.tree import export_text

tree_rules = export_text(
    tree,
    feature_names=tree_features
)

print(tree_rules)

|--- impressions_90d <= 5.50
|   |--- impressions_90d <= 3.50
|   |   |--- avg_position <= 11.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  11.15
|   |   |   |--- class: 0
|   |--- impressions_90d >  3.50
|   |   |--- content_age_days <= 97.00
|   |   |   |--- class: 1
|   |   |--- content_age_days >  97.00
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 356.50
|   |   |--- clicks_90d <= 27.50
|   |   |   |--- class: 1
|   |   |--- clicks_90d >  27.50
|   |   |   |--- class: 0
|   |--- content_age_days >  356.50
|   |   |--- avg_position <= 23.55
|   |   |   |--- class: 0
|   |   |--- avg_position >  23.55
|   |   |   |--- class: 0



In [ ]:
# Section 3G — Evaluate the ML-07 baseline on the same test split

# Recreate the Week-4 baseline rule
baseline_test = df_test_model.copy()

baseline_test["baseline_score"] = (
    (baseline_test["impressions_90d"] >= 500).astype(int) * 2
    + (baseline_test["days_since_last_update"] >= 90).astype(int) * 2
    + (baseline_test["ctr"] <= 0.30).astype(int)
)

# Baseline predicts declining when score is positive
baseline_pred = (baseline_test["baseline_score"] > 0).astype(int)

baseline_accuracy = accuracy_score(
    baseline_test["is_declining_label"],
    baseline_pred
)

baseline_precision = precision_score(
    baseline_test["is_declining_label"],
    baseline_pred,
    zero_division=0
)

baseline_recall = recall_score(
    baseline_test["is_declining_label"],
    baseline_pred,
    zero_division=0
)

baseline_f1 = f1_score(
    baseline_test["is_declining_label"],
    baseline_pred,
    zero_division=0
)

print("ML-07 baseline on the same test split:")
print("Accuracy:", round(baseline_accuracy, 4))
print("Precision:", round(baseline_precision, 4))
print("Recall:", round(baseline_recall, 4))
print("F1:", round(baseline_f1, 4))

ML-07 baseline on the same test split:
Accuracy: 0.4933
Precision: 0.5023
Recall: 0.9108
F1: 0.6475


### Model vs baseline

The Decision Tree had higher accuracy and precision than the ML-07 baseline on the held-out client test split. However, the baseline had higher recall and F1. Therefore, the Decision Tree is not an overall replacement for the baseline. The tree was more selective, while the baseline identified a larger share of the declining items.

In [ ]:
# Section 3 — Final model vs baseline comparison

comparison = pd.DataFrame({
    "Model": ["ML-07 Baseline", "Decision Tree"],
    "Accuracy": [0.4933, 0.5776],
    "Precision": [0.5023, 0.5855],
    "Recall": [0.9108, 0.5938],
    "F1": [0.6475, 0.5896]
})

display(comparison)

,Model,Accuracy,Precision,Recall,F1
0,ML-07 Baseline,0.4933,0.5023,0.9108,0.6475
1,Decision Tree,0.5776,0.5855,0.5938,0.5896


### 4. Errors and interpretation

The Decision Tree was correct on 3,560 of 6,163 test rows, with 1,324 false positives and 1,279 false negatives.

The model relied mainly on impressions_90d (importance 0.595) and content_age_days (0.238), followed by avg_position (0.086) and clicks_90d (0.081).

The false negatives had higher average impressions, clicks, sessions, and content age than correct predictions. This suggests that the tree missed some declining pages that still had substantial historical visibility.

The false positives had much lower average impressions, clicks, and sessions than correct predictions. This suggests that some lower-volume pages were classified as declining even though they were not labeled declining.

These are observed patterns in the held-out test set and are useful for decision-support, not causal conclusions.

In [ ]:
# Section 4A — Feature importance

feature_importance = pd.DataFrame({
    "feature": tree_features,
    "importance": tree.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(feature_importance)

,feature,importance
0,impressions_90d,0.594798
8,content_age_days,0.237926
4,avg_position,0.086064
1,clicks_90d,0.081212
3,ctr,0.000000
2,sessions_90d,0.000000
5,engagement_rate,0.000000
6,scroll_rate,0.000000
7,days_since_last_update,0.000000
9,word_count,0.000000


In [ ]:
# Section 4B — Error analysis

error_analysis = df_test_model[
    ["content_id", "is_declining_label"]
].copy()

error_analysis["prediction"] = tree_pred

error_analysis["error_type"] = "Correct"

error_analysis.loc[
    (error_analysis["is_declining_label"] == 1) &
    (error_analysis["prediction"] == 0),
    "error_type"
] = "False Negative"

error_analysis.loc[
    (error_analysis["is_declining_label"] == 0) &
    (error_analysis["prediction"] == 1),
    "error_type"
] = "False Positive"

print("Error counts:")
print(error_analysis["error_type"].value_counts())

Error counts:
error_type
Correct           3560
False Positive    1324
False Negative    1279
Name: count, dtype: int64


In [ ]:
# Compare characteristics of correct and incorrect predictions

error_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]

error_data = df_test_model[
    error_features + ["is_declining_label"]
].copy()

error_data["prediction"] = tree_pred

error_data["error_type"] = "Correct"

error_data.loc[
    (error_data["is_declining_label"] == 1) &
    (error_data["prediction"] == 0),
    "error_type"
] = "False Negative"

error_data.loc[
    (error_data["is_declining_label"] == 0) &
    (error_data["prediction"] == 1),
    "error_type"
] = "False Positive"

display(
    error_data
    .groupby("error_type")[error_features]
    .mean()
    .round(2)
)

,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,content_age_days,days_since_last_update
error_type,,,,,,,
Correct,4267.28,12.26,14.45,0.29,14.63,282.60,33.11
False Negative,6717.65,19.81,20.31,0.19,16.43,417.30,33.82
False Positive,1395.89,2.64,5.85,0.39,17.93,202.08,40.15


### Self-check

- Every section is filled with both markdown reasoning and supporting code.
- The notebook runs from top to bottom without errors.
- The train/test split was grouped by client, with no client overlap.
- The Decision Tree was trained only on allowed features and did not use trend_direction or trend_pct as features.
- Missing values were handled using training-data median imputation.
- The model was evaluated on the held-out test clients.
- The Decision Tree was compared with the Week-4 baseline using the same test split and metrics.
- The results are reported as observed and measured results rather than causal claims.
- No client names, URLs, or private queries are included.